<a href="https://colab.research.google.com/github/Joselala123/Portugal_House_DataCleaning_EDA_PredictiveModel/blob/main/Portugal_House_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Real Estate Listings in Portugal - Andrade José <br>**
<h2>Data Cleaning, EDA and Predictive Model</h2>


##Dataset Cleaning

###Libraries and dataset loading

In [36]:
#Libraries for data analysis and visualization
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [37]:
# pandas library config
pd.set_option('display.max_rows', None)
pd.set_option("display.max_columns", None)

In [38]:
# Load of dataset

file_id = "1Zb1wlWyr7DQOftR-yf8UsrsK71Gd6bQv"

url = f"https://drive.google.com/uc?export=download&id={file_id}"

df = pd.read_csv(url, low_memory=False)

print(f"Dimensión original: {df.shape}")

Dimensión original: (179001, 23)


###First look of the dataset

In [39]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 179001 entries, 0 to 179000
Data columns (total 23 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   Price                  178547 non-null  float64
 1   District               179001 non-null  object 
 2   City                   179001 non-null  object 
 3   Town                   178999 non-null  object 
 4   Type                   178985 non-null  object 
 5   EnergyCertificate      178987 non-null  object 
 6   Floor                  27929 non-null   object 
 7   NumberOfBathrooms      162517 non-null  float64
 8   Parking                178969 non-null  float64
 9   TotalArea              164059 non-null  float64
 10  ConstructionYear       117542 non-null  float64
 11  EnergyEfficiencyLevel  110754 non-null  object 
 12  PublishDate            71260 non-null   object 
 13  Garage                 110754 non-null  object 
 14  Elevator               178969 non-nu

In [40]:
df.head()

,Price,District,City,Town,Type,EnergyCertificate,Floor,NumberOfBathrooms,Parking,TotalArea,ConstructionYear,EnergyEfficiencyLevel,PublishDate,Garage,Elevator,ElectricCarsCharging,TotalRooms,NumberOfBedrooms,NumberOfWC,ConservationStatus,LivingArea,LotSize,BuiltArea
0,250000.0,Faro,São Brás de Alportel,São Brás de Alportel,Apartment,A+,1st Floor,0.0,1.0,114.0,NaN,NaN,NaN,NaN,True,NaN,2.0,NaN,NaN,NaN,114.0,NaN,NaN
1,158000.0,Faro,Portimão,Portimão,Apartment,D,2nd Floor,2.0,1.0,21953.0,1993.0,NaN,NaN,NaN,True,NaN,1.0,NaN,NaN,NaN,42.0,NaN,48.0
2,19000.0,Faro,Lagoa (Algarve),Estômbar e Parchal,Land,NC,NaN,0.0,0.0,3710.0,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,3710.0,NaN,3710.0
3,250000.0,Faro,São Brás de Alportel,São Brás de Alportel,Apartment,A+,1st Floor,2.0,1.0,115.0,NaN,NaN,NaN,NaN,True,NaN,2.0,NaN,NaN,NaN,115.0,NaN,NaN
4,2950000.0,Faro,Albufeira,Albufeira e Olhos de Água,Building,NC,NaN,0.0,0.0,265.0,1998.0,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,406.0,NaN,465.0


###1. Basic normalization and house selection

In [41]:
# Normalize the column names
df.columns = df.columns.str.strip()

# Normalize string variables
column_list=["Type", "District", "Town", "City", "EnergyEfficiencyLevel", "EnergyCertificate"]

for col in column_list:
  df[col] = df[col].str.title()

# Assign the correct data type to each variable

# String
text_cols = ["District", "City", "Town", "Type", "EnergyCertificate", "Floor","EnergyEfficiencyLevel", "EnergyEfficiencyLevel","PublishDate","ConservationStatus"]

for col in text_cols:
    df[col] = (df[col].astype("string").str.strip())

# Int
int_cols= ["NumberOfBathrooms","ConstructionYear","NumberOfBedrooms","NumberOfWC", "Parking"]

for col in int_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

# Bool
bool_cols=["Garage", "Elevator", "ElectricCarsCharging"]

for col in bool_cols:
  df[col] = df[col].astype("boolean")

# Timestamp
df["PublishDate"] = pd.to_datetime(
    df["PublishDate"],
    format="mixed",
    errors="coerce"
)

# Filter only houses
df = df[df["Type"].eq("House")].copy()

# Drop Type column
df.drop(columns=["Type"], inplace=True)

# register the original position of each observation
df["original_row_id"] = df.index

columns = ["original_row_id"] + [col for col in df.columns if col != "original_row_id"]

df = df[columns].copy()

#Add an identifier to each observation
df.insert(len(df.columns), "ID", range(1, len(df) + 1))
df.set_index('ID', inplace=True)


print(f"Number of rows: {len(df)}")


Number of rows: 48744


###2. Missing-value inspection

In [42]:
#Function to display missing values and the percentage they represent
def show_nulls(df):
  nulls = (
      pd.DataFrame({"nMissing": df.isna().sum(), "percentage": (df.isna().sum()/len(df))*100}).sort_values("percentage", ascending=False)
  )
  print("Count of null values for each variable and its percentage:")
  display(nulls)

In [43]:
show_nulls(df)

Count of null values for each variable and its percentage:


,nMissing,percentage
Floor,47677,97.811013
ConservationStatus,39233,80.487855
PublishDate,29164,59.830954
NumberOfWC,26527,54.421057
LotSize,25801,52.931643
BuiltArea,24464,50.188741
NumberOfBedrooms,18720,38.404727
ElectricCarsCharging,18717,38.398572
EnergyEfficiencyLevel,18717,38.398572
Garage,18717,38.398572


In [44]:
#Variables with a large number of missing values are removed.
#df.drop(columns=["Floor","ConservationStatus", "BuiltArea"], inplace=True)

In [45]:
show_nulls(df)

Count of null values for each variable and its percentage:


,nMissing,percentage
Floor,47677,97.811013
ConservationStatus,39233,80.487855
PublishDate,29164,59.830954
NumberOfWC,26527,54.421057
LotSize,25801,52.931643
BuiltArea,24464,50.188741
NumberOfBedrooms,18720,38.404727
ElectricCarsCharging,18717,38.398572
EnergyEfficiencyLevel,18717,38.398572
Garage,18717,38.398572


#### 2.1 Missing-value pattern
A missing value pattern was identified, 18.717 observations have missing values in the same variables, including Garage, ElectricCarsCharging, NumberOfBedrooms, NumberOfWC, LotSize, and PublishDate. Since the dataset was built through web scraping:
*   One possible explanation is that **different scraping sources provide different characteristics.**

It is important to mention that linear models do not accept missing values, so if the previous statement is true, part of the dataset could be excluded and, potentially, representation from one of the scraping sources could be lost, biasing the model.

In [46]:
# Variables that belong to the pattern
pattern_columns = [
    "EnergyEfficiencyLevel",
    "PublishDate",
    "Garage",
    "ElectricCarsCharging",
    "NumberOfBedrooms",
    "NumberOfWC",
    "LotSize"
]

In [47]:
df[df[pattern_columns].isna().all(axis=1)].head()

,original_row_id,Price,District,City,Town,EnergyCertificate,Floor,NumberOfBathrooms,Parking,TotalArea,ConstructionYear,EnergyEfficiencyLevel,PublishDate,Garage,Elevator,ElectricCarsCharging,TotalRooms,NumberOfBedrooms,NumberOfWC,ConservationStatus,LivingArea,LotSize,BuiltArea
ID,,,,,,,,,,,,,,,,,,,,,,,
1,10,235000.0,Faro,Faro,Faro (Sé E São Pedro),Nc,<NA>,2,0,63.0,<NA>,<NA>,NaT,<NA>,False,<NA>,1.0,<NA>,<NA>,<NA>,63.0,NaN,NaN
2,26,135000.0,Faro,Vila Real De Santo António,Vila Nova De Cacela,Nc,<NA>,1,1,199.0,1951,<NA>,NaT,<NA>,False,<NA>,1.0,<NA>,<NA>,<NA>,35.0,NaN,43.0
3,29,199000.0,Faro,Portimão,Portimão,C,<NA>,2,0,82.0,2023,<NA>,NaT,<NA>,False,<NA>,2.0,<NA>,<NA>,<NA>,82.0,NaN,NaN
4,32,175000.0,Faro,Faro,Faro (Sé E São Pedro),Nc,<NA>,2,0,78.0,1950,<NA>,NaT,<NA>,False,<NA>,4.0,<NA>,<NA>,<NA>,78.0,NaN,NaN
5,33,1485000.0,Faro,Loulé,Quarteira,D,Ground Floor,5,2,472.0,2004,<NA>,NaT,<NA>,False,<NA>,4.0,<NA>,<NA>,<NA>,414.0,NaN,NaN


In [48]:
df[df[pattern_columns].isna().all(axis=1)].tail()

,original_row_id,Price,District,City,Town,EnergyCertificate,Floor,NumberOfBathrooms,Parking,TotalArea,ConstructionYear,EnergyEfficiencyLevel,PublishDate,Garage,Elevator,ElectricCarsCharging,TotalRooms,NumberOfBedrooms,NumberOfWC,ConservationStatus,LivingArea,LotSize,BuiltArea
ID,,,,,,,,,,,,,,,,,,,,,,,
18717,68239,295000.0,Setúbal,Montijo,Pegões,E,<NA>,1,3,86.0,1982,<NA>,NaT,<NA>,False,<NA>,3.0,<NA>,<NA>,<NA>,106.0,NaN,NaN
18718,68242,645000.0,Setúbal,Palmela,Quinta Do Anjo,A,<NA>,3,1,223.0,2002,<NA>,NaT,<NA>,False,<NA>,4.0,<NA>,<NA>,<NA>,169.0,NaN,NaN
18719,68245,55000.0,Viana Do Castelo,Arcos De Valdevez,Sabadim,F,<NA>,0,0,1400.0,1937,<NA>,NaT,<NA>,False,<NA>,1.0,<NA>,<NA>,<NA>,40.0,NaN,80.0
18720,68249,62000.0,Vila Real,Santa Marta De Penaguião,Lobrigos (São Miguel E São João Baptista) E Sa...,Nc,<NA>,1,0,30.0,1997,<NA>,NaT,<NA>,False,<NA>,2.0,<NA>,<NA>,<NA>,30.0,NaN,60.0
18721,68251,75000.0,Vila Real,Vila Real,Vila Real,Nc,<NA>,2,0,56.0,1987,<NA>,NaT,<NA>,False,<NA>,2.0,<NA>,<NA>,<NA>,46.0,NaN,96.0


By observing the first and last 5 observations, it is possible that the missing value pattern appears sequentially across observations.

In [49]:
indices = df.index[df[pattern_columns].isna().all(axis=1)].to_series()

# Identify consecutive blocks
blocks = (
    indices
    .groupby(indices.diff().ne(1).cumsum())
    .agg(["min", "max", "count"])
)

display(blocks)

,min,max,count
ID,,,
1,1,18699,18699
2,18702,18713,12
3,18716,18721,6


Across most of the interval: [1, 18721], a consistent missing value pattern is observed, with a few exceptions. This continuity suggests that the observations in this interval could come from the same scraping process, where the source did not contain the variables that appear as missing.

In [50]:
#Values that do not follow the pattern
pd.concat([df.iloc[18699:18701], df.iloc[18713:18715]], axis=0)

,original_row_id,Price,District,City,Town,EnergyCertificate,Floor,NumberOfBathrooms,Parking,TotalArea,ConstructionYear,EnergyEfficiencyLevel,PublishDate,Garage,Elevator,ElectricCarsCharging,TotalRooms,NumberOfBedrooms,NumberOfWC,ConservationStatus,LivingArea,LotSize,BuiltArea
ID,,,,,,,,,,,,,,,,,,,,,,,
18700,68195,62000.0,Aveiro,Ovar,Válega,F,<NA>,1,1,336.0,1950,F,2024-10-21 15:56:26.597,False,False,False,NaN,2,<NA>,<NA>,300.0,NaN,336.0
18701,68197,220000.0,Aveiro,Oliveira Do Bairro,Palhaça,F,<NA>,1,0,154.0,1993,F,2024-10-18 14:27:49.537,True,False,False,NaN,4,1,<NA>,606.0,5982.0,NaN
18714,68231,37000.0,Aveiro,Ílhavo,São Salvador,Nc,<NA>,2,0,140.0,1950,Nc,2024-10-18 16:15:40.007,False,False,False,NaN,3,2,<NA>,90.0,315.0,140.0
18715,68233,250000.0,Aveiro,Murtosa,Monte,C,<NA>,4,1,68.0,1951,C,2024-10-11 15:39:50.397,False,False,False,NaN,3,1,Like new,55.0,NaN,93.0


A new column is created to identify this new sub-block.

In [51]:
# It is enough for data to be present in any of these fields
df["source"] = df[pattern_columns].notna().any(axis=1).map({
    False: "fuente_A",
    True: "unknown"
})

# Check the classification
summary = (
    df["source"]
    .value_counts()
    .to_frame("records")
    .sort_index()
)

summary["percentage"] = (
    summary["records"] / len(df) * 100
).round(2)

print(summary)

          records  percentage
source                       
fuente_A    18717        38.4
unknown     30027        61.6


Around 40% of the filtered dataset belongs to `fuente_A`.

### 3. Duplicates

In [52]:
dfDuplicates=df.copy()

In [53]:
print(f"There are {dfDuplicates[df.duplicated(keep=False)].shape[0]} registered duplicates.")

There are 0 registered duplicates.


In [54]:
columns = dfDuplicates.columns.drop("original_row_id", "source").to_list()

groups = dfDuplicates.groupby(columns,dropna=False)

dfDuplicates["DuplicateCount"] = groups.transform("size")
dfDuplicates["DuplicateGroup"] = groups.ngroup() + 1

In [55]:
dfDuplicates["DuplicateCount"].value_counts().sort_index(ascending=True)

,count
DuplicateCount,
1,44206
2,3412
3,513
4,176
5,60
6,48
7,42
8,32
9,27


The table above shows that some records have more than one associated duplicate, there is even a group of 29 records that are identical in the variables analyzed.
* One reasonable explanation could be that they belong to the same house design published by the same person, organization, among others. This could occur in residential developments.

Other explanations:
* Repeatedly capturing a listing that remains published.
* The same property appearing in different scraping sources.
* Concatenating extractions without controlling for records that had already been incorporated.

In [56]:
dfDuplicates[dfDuplicates["PublishDate"].notna() & dfDuplicates["DuplicateCount"]>2]

,original_row_id,Price,District,City,Town,EnergyCertificate,Floor,NumberOfBathrooms,Parking,TotalArea,ConstructionYear,EnergyEfficiencyLevel,PublishDate,Garage,Elevator,ElectricCarsCharging,TotalRooms,NumberOfBedrooms,NumberOfWC,ConservationStatus,LivingArea,LotSize,BuiltArea,source,DuplicateCount,DuplicateGroup
ID,,,,,,,,,,,,,,,,,,,,,,,,,,


In [57]:
#Removal of exact duplicates
df[df.duplicated(keep=False) & df["PublishDate"].notna()].head()

#df.drop_duplicates(inplace=True, keep="first")

,original_row_id,Price,District,City,Town,EnergyCertificate,Floor,NumberOfBathrooms,Parking,TotalArea,ConstructionYear,EnergyEfficiencyLevel,PublishDate,Garage,Elevator,ElectricCarsCharging,TotalRooms,NumberOfBedrooms,NumberOfWC,ConservationStatus,LivingArea,LotSize,BuiltArea,source
ID,,,,,,,,,,,,,,,,,,,,,,,,


#### Conservative detection of possible duplicates

Multiple properties were observed with the same price, area, number of bathrooms, and location. However, these matches are not sufficient to state that they correspond to the same property, since different properties belonging to the same residential development may share floor plans and practically identical characteristics.

For this reason, the variables `Price`, `LivingArea`, `TotalArea`, `NumberOfBathrooms`, `District`, `City`, and `Town` are used only to form candidate blocks.

Within each block, observations are compared in pairs using additional relatively stable property attributes, such as construction year, energy certificate, elevator, built area, lot size, and number of WCs. Room information available across the different scraping schemas is also used.

A pair is considered a possible high-confidence duplicate when:

- it presents different missing-information patterns, consistent with records coming from different scraping schemas;
- it matches on at least three additional available attributes;
- it presents no contradictions in those attributes;
- each observation has only one compatible candidate within the group.

This last criterion avoids automatically consolidating different properties that, for example, belong to the same residential development and share similar characteristics.

For pairs identified with high confidence, the observation with the most recent `PublishDate` is kept as the base record when both dates are available. Otherwise, the observation with fewer missing values is selected. The missing values in the base row are then completed with information available in the other observation, without overwriting already observed values.

Ambiguous cases are kept unchanged, prioritizing the avoidance of removing potentially different properties.

In [58]:
from itertools import combinations
from collections import Counter

# These variables are used only to generate groups of similar properties,
# not to directly claim that they are duplicates.

column_list = [
    "Price",
    "LivingArea",
    "TotalArea",
    "NumberOfBathrooms",
    "District",
    "City",
    "Town"
]

possible_duplicates = df[ df[column_list].notna().all(axis=1) & df.duplicated(subset=column_list, keep=False)].copy()

# Create candidate groups
possible_duplicates["group"] = (possible_duplicates.groupby(column_list).ngroup())

# Relatively stable variables for comparing properties
stable_vars = [
    "ConstructionYear",
    "EnergyCertificate",
    "Elevator",
    "LotSize",
    "NumberOfWC"
]


# Variables whose NaN pattern helps distinguish
# different scraping schemas
schema_vars = [
    "TotalRooms",
    "NumberOfBedrooms",
    "NumberOfWC",
    "Garage",
    "ElectricCarsCharging",
    "PublishDate",
    "LotSize",
]


# Minimum number of additional matches
MIN_MATCHES = 3


def rooms(row):

    if pd.notna(row["NumberOfBedrooms"]):
        return row["NumberOfBedrooms"]

    return row["TotalRooms"]


def is_match(row1, row2):

    matches = 0
    conflicts = 0

    # --------------------------------------------------
    # 1. Require a different information pattern
    # --------------------------------------------------

    pattern1 = tuple(
        row1[col] is not None and pd.notna(row1[col])
        for col in schema_vars
    )

    pattern2 = tuple(
        row2[col] is not None and pd.notna(row2[col])
        for col in schema_vars
    )

    if pattern1 == pattern2:
        return False


    # --------------------------------------------------
    # 2. Compare stable variables
    # --------------------------------------------------

    for col in stable_vars:

        value1 = row1[col]
        value2 = row2[col]

        # Only compare if both values are available
        if pd.notna(value1) and pd.notna(value2):

            if value1 == value2:
                matches += 1

            else:
                conflicts += 1


    # --------------------------------------------------
    # 3. Compare room information
    # --------------------------------------------------

    rooms1 = rooms(row1)
    rooms2 = rooms(row2)

    if pd.notna(rooms1) and pd.notna(rooms2):

        if rooms1 == rooms2:
            matches += 1

        else:
            conflicts += 1


    return (
        conflicts == 0
        and matches >= MIN_MATCHES
    )


# ============================================================
# COMPARE ROWS WITHIN EACH GROUP
# ============================================================

pairs = []

for group_id, group in possible_duplicates.groupby("group"):

    for i, j in combinations(group.index, 2):

        row1 = possible_duplicates.loc[i]
        row2 = possible_duplicates.loc[j]

        if is_match(row1, row2):

            pairs.append(
                (i, j, group_id)
            )


# ============================================================
# KEEP ONLY UNIQUE MATCHES
# ============================================================

# Count how many possible matches each observation has
count = Counter()

for i, j, group_id in pairs:

    count[i] += 1
    count[j] += 1


# Only consolidate A-B if A matches only B
# and B matches only A
unique_pairs = [
    (i, j, group_id)
    for i, j, group_id in pairs
    if count[i] == 1 and count[j] == 1
]


print(
    "Grupos candidatos:",
    possible_duplicates["group"].nunique()
)

print(
    "Pares de alta confianza:",
    len(unique_pairs)
)

Grupos candidatos: 4465
Pares de alta confianza: 1984


In [59]:
for i, j, group_id in unique_pairs[:10]:

    print("Grupo:", group_id)

    display(
        df.loc[
            [i, j]
        ]
    )

Grupo: 0


,original_row_id,Price,District,City,Town,EnergyCertificate,Floor,NumberOfBathrooms,Parking,TotalArea,ConstructionYear,EnergyEfficiencyLevel,PublishDate,Garage,Elevator,ElectricCarsCharging,TotalRooms,NumberOfBedrooms,NumberOfWC,ConservationStatus,LivingArea,LotSize,BuiltArea,source
ID,,,,,,,,,,,,,,,,,,,,,,,,
5045,17854,2300.0,Castelo Branco,Castelo Branco,Castelo Branco,Nc,<NA>,0,0,29.0,<NA>,<NA>,NaT,<NA>,False,<NA>,0.0,<NA>,<NA>,<NA>,29.0,NaN,NaN,fuente_A
21877,77693,2300.0,Castelo Branco,Castelo Branco,Castelo Branco,Nc,<NA>,0,0,29.0,1900,Nc,NaT,False,False,False,NaN,0,0,<NA>,29.0,29.0,NaN,unknown


Grupo: 1


,original_row_id,Price,District,City,Town,EnergyCertificate,Floor,NumberOfBathrooms,Parking,TotalArea,ConstructionYear,EnergyEfficiencyLevel,PublishDate,Garage,Elevator,ElectricCarsCharging,TotalRooms,NumberOfBedrooms,NumberOfWC,ConservationStatus,LivingArea,LotSize,BuiltArea,source
ID,,,,,,,,,,,,,,,,,,,,,,,,
5044,17840,2400.0,Castelo Branco,Castelo Branco,Castelo Branco,Nc,<NA>,0,0,31.0,<NA>,<NA>,NaT,<NA>,False,<NA>,0.0,<NA>,<NA>,<NA>,31.0,NaN,NaN,fuente_A
21844,77516,2400.0,Castelo Branco,Castelo Branco,Castelo Branco,Nc,<NA>,0,0,31.0,1900,Nc,2021-02-06 10:55:01.667,False,False,False,1.0,0,0,<NA>,31.0,31.0,NaN,unknown


Grupo: 2


...

,original_row_id,Price,District,City,Town,EnergyCertificate,Floor,NumberOfBathrooms,Parking,TotalArea,ConstructionYear,EnergyEfficiencyLevel,PublishDate,Garage,Elevator,ElectricCarsCharging,TotalRooms,NumberOfBedrooms,NumberOfWC,ConservationStatus,LivingArea,LotSize,BuiltArea,source
ID,,,,,,,,,,,,,,,,,,,,,,,,
17693,64620,7000.0,Vila Real,Montalegre,Ferral,Nc,<NA>,0,0,59.0,2012,<NA>,NaT,<NA>,False,<NA>,0.0,<NA>,<NA>,<NA>,119.0,NaN,NaN,fuente_A
30868,113832,7000.0,Vila Real,Montalegre,Ferral,Nc,<NA>,0,0,59.0,2012,Nc,2024-09-11 10:18:57.307,False,False,False,NaN,0,0,<NA>,119.0,NaN,NaN,unknown


Grupo: 9


,original_row_id,Price,District,City,Town,EnergyCertificate,Floor,NumberOfBathrooms,Parking,TotalArea,ConstructionYear,EnergyEfficiencyLevel,PublishDate,Garage,Elevator,ElectricCarsCharging,TotalRooms,NumberOfBedrooms,NumberOfWC,ConservationStatus,LivingArea,LotSize,BuiltArea,source
ID,,,,,,,,,,,,,,,,,,,,,,,,
5214,18562,7250.0,Castelo Branco,Oleiros,Oleiros - Amieira,Nc,<NA>,1,0,84.0,1937,<NA>,NaT,<NA>,False,<NA>,2.0,<NA>,<NA>,<NA>,84.0,NaN,NaN,fuente_A
21696,76996,7250.0,Castelo Branco,Oleiros,Oleiros - Amieira,Nc,<NA>,1,0,84.0,1937,Nc,NaT,False,False,False,NaN,2,1,Needs renovation,84.0,NaN,NaN,unknown


Grupo: 10


,original_row_id,Price,District,City,Town,EnergyCertificate,Floor,NumberOfBathrooms,Parking,TotalArea,ConstructionYear,EnergyEfficiencyLevel,PublishDate,Garage,Elevator,ElectricCarsCharging,TotalRooms,NumberOfBedrooms,NumberOfWC,ConservationStatus,LivingArea,LotSize,BuiltArea,source
ID,,,,,,,,,,,,,,,,,,,,,,,,
18114,66090,7500.0,Castelo Branco,Castelo Branco,Escalos De Baixo E Mata,Nc,<NA>,0,0,17.0,1937,<NA>,NaT,<NA>,False,<NA>,1.0,<NA>,<NA>,<NA>,15.0,NaN,NaN,fuente_A
21374,76114,7500.0,Castelo Branco,Castelo Branco,Escalos De Baixo E Mata,Nc,<NA>,0,0,17.0,1937,Nc,2024-09-18 16:39:04.567,False,False,False,2.0,1,0,<NA>,15.0,NaN,NaN,unknown


####Second attempt

In [60]:
# list of variables to identify possible duplicates
column_list = [
    "Price",
    "LivingArea",
    "TotalArea",
    "NumberOfBathrooms",
    "District",
    "City",
    "Town"
]

# Detect candidates
possible_duplicates = df[ df[column_list].notna().all(axis=1) & df.duplicated(subset=column_list, keep=False)].copy()

# Create group
possible_duplicates["group"] = (possible_duplicates.groupby(column_list, dropna=False).ngroup())

# Variables that are not part of the list
non_key_cols = [
    col for col in possible_duplicates.columns
      if col not in column_list + ["group", "PublishDate", "original_row_id", "schema"]
]

# Detect conflicts
duplicates = (possible_duplicates.groupby("group")[non_key_cols].nunique(dropna=True))

compatible_groups = duplicates[(duplicates <= 1).all(axis=1)].index

conflicting_groups = duplicates[(duplicates > 1).any(axis=1)].index

print(f"Grupos compatibles: {len(compatible_groups)}")
print(f"Grupos conflituales: {len(conflicting_groups)}")

Grupos compatibles: 1994
Grupos conflituales: 2471


#### Logic for handling duplicates

During the exploration of the missing-value pattern, groups of observations with exact matches in variables such as price, area, number of bathrooms, and location were identified.

Inspection showed that many of these groups could correspond to the same property recorded multiple times.
**len(compatible_groups)** compatible groups were found, meaning that a combination of both "duplicate" observations can be performed without problems.

On the other hand, **len(conflicting_groups)** conflicting groups were found, meaning that they do show differences in the variables not considered when searching for exact matches.
<br></br>
In several cases, observations differ only in some attributes or in the publication date. For this reason, instead of arbitrarily removing repeated records, the following logic is applied:

1. If all observations in the group have `PublishDate`, the observation with the most recent date is kept. If there is a tie in the date, the row with fewer missing values is selected.
2. If any observation in the group does not have `PublishDate`, the row with fewer missing values is selected directly, since it is not possible to reliably establish which observation is temporally more recent.
3. Once the base row is selected, its observed values are not modified. Missing values are completed using information from the other observations in the group only when there is a single consistent non-null value for that variable.
4. If contradictory values exist for a variable that is missing in the base row, the value remains missing instead of selecting one arbitrarily.

This procedure makes it possible to preserve a real observation as the main record and recover complementary information without overwriting observed values or resolving conflicts through arbitrary decisions.

In [61]:
def consolidate(group):

    # Count NaN values excluding PublishDate and group
    cols = group.columns.difference(["group", "PublishDate"])
    nulls = group[cols].isna().sum(axis=1)

    # If all have a date -> choose the most recent one
    if group["PublishDate"].notna().all():
        max_date = group["PublishDate"].max()
        candidates = group[group["PublishDate"] == max_date]

        # If there is a tie in date -> choose the most complete one
        base = candidates.loc[
            nulls.loc[candidates.index].idxmin()
        ].copy()

    # If any row has no date -> choose the most complete one
    else:
        base = group.loc[nulls.idxmin()].copy()

    # Fill only the NaN values in the base row
    for col in cols:

        if pd.isna(base[col]):

            values = group[col].dropna().unique()

            # Fill only if there is no conflict
            if len(values) == 1:
                base[col] = values[0]

    return base


# One row per group
merged = (
    possible_duplicates
    .groupby("group", group_keys=False)
    .apply(consolidate)
    .reset_index(drop=True)
)


# Remove the original rows that were consolidated
df_remaining = df.drop(
    index=possible_duplicates.index
)


# Add the consolidated rows
df = pd.concat(
    [
        df_remaining,
        merged.drop(columns="group", errors="ignore")
    ],
    ignore_index=True
)

print("Shape final:", df.shape)

Shape final: (42623, 24)


/tmp/ipykernel_1700/558361574.py:39: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(consolidate)


### 4. Validation of impossible values

In [62]:
#Simple inspection of minimum and maximum values
df.describe().loc[['min', 'max']]

,original_row_id,Price,NumberOfBathrooms,Parking,TotalArea,PublishDate,TotalRooms,LivingArea,LotSize,BuiltArea
min,26.0,1.000000e+00,-1.0,0.0,-270.0,2018-05-02 00:00:00,0.0,0.0,0.0,0.0
max,178995.0,1.380000e+09,3020.0,3.0,5525000.0,2026-09-02 20:30:15.870000,2751.0,470300.0,40672000.0,470300.0


In [63]:
# Price
df = df[(df["Price"] > 0) & (df["Price"] <= 100_000_000)].copy()

# Remove observation with a value of 1.
df.drop(
    index=df[df["Price"] == 1].index,
    inplace=True
)

In [64]:
# Areas
area_cols = ["LivingArea", "TotalArea", "LotSize"]

for col in area_cols:
    invalid = (df[col].notna() & (df[col] <= 5)) #An area variable with a value of 5 or less is considered impossible

    df.loc[invalid, col] = np.nan

In [65]:
#Illogical areas (LivingArea > TotalArea)

(((df["LivingArea"] > df["TotalArea"])).sum())

np.int64(5524)

LivingArea and TotalArea do not conceptually represent the same thing; however, there are 4,247 observations that appear this way. <br>
We will visualize how the difference between LivingArea and TotalArea behaves.

In [66]:
mask = (df["LivingArea"].notna() & df["TotalArea"].notna() & (df["LivingArea"] > df["TotalArea"]))

area_review = df.loc[mask, ["LivingArea", "TotalArea"]].copy()

area_review["difference"] = (area_review["LivingArea"] - area_review["TotalArea"])

area_review["ratio"] = (area_review["LivingArea"] / area_review["TotalArea"])

area_review[["difference", "ratio"]].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95])

,difference,ratio
count,5524.000000,5524.000000
mean,333.495293,3.091622
std,5741.180363,32.053783
min,1.000000,1.000812
10%,14.000000,1.108523
25%,30.000000,1.250000
50%,61.000000,1.564752
75%,115.000000,2.000000
90%,196.000000,2.590107
95%,288.000000,3.146482


There are considerable differences between LivingArea and TotalArea in a relevant proportion of observations. These inconsistencies could be due to differences in how the variables are defined across scraping sources, recording errors, user confusion, among other reasons. <br>
Since it is not possible to determine which of the two variables is correct in each case, the observations will be retained and an indicator variable will be created to identify cases where LivingArea > TotalArea. This variable will allow further analysis and could help model performance.

In [67]:
mask = (df["TotalArea"].notna() & df["LivingArea"].notna())

df["AreaInconsistency"] = pd.Series(pd.NA, index=df.index, dtype="Int64")

df.loc[mask, "AreaInconsistency"] = (df.loc[mask, "LivingArea"]> df.loc[mask, "TotalArea"]).astype(int)

In [68]:
df[['TotalArea','LivingArea',"AreaInconsistency"]].head()

,TotalArea,LivingArea,AreaInconsistency
0,199.0,35.0,0
1,82.0,82.0,0
2,472.0,414.0,0
3,140.0,158.0,1
4,7576.0,188.0,0


In [69]:
# Count variables
count_cols = [
    "NumberOfBathrooms",
    "NumberOfBedrooms",
    "NumberOfWC",
    "TotalRooms",
]

for col in count_cols:
    invalid = ((df[col].notna())& (df[col] < 0))

    if invalid.any():
        print(f"{col}: {invalid.sum():,} valores negativos convertidos a NaN.")
        df.loc[invalid, col] = np.nan

#Thresholds above which an observation is considered impossible.
limits = {                   #Interpretation:
    "NumberOfBathrooms": 30,  #Number of bathrooms >30
    "NumberOfWC": 30,
    "NumberOfBedrooms": 50,
    "TotalRooms": 100
}

for col, upper in limits.items():
    invalid = (df[col].notna() & (df[col] > upper))

    print(f"{col} > {upper}: {invalid.sum():,} valores convertidos a NaN.")

    df.loc[invalid, col] = np.nan


NumberOfBathrooms: 1 valores negativos convertidos a NaN.
NumberOfBathrooms > 30: 2 valores convertidos a NaN.
NumberOfWC > 30: 1 valores convertidos a NaN.
NumberOfBedrooms > 50: 0 valores convertidos a NaN.
TotalRooms > 100: 1 valores convertidos a NaN.


### 5. Cleaning summary

In [70]:
print(f"Casas finales después de limpieza: {len(df)}")
print(f"Variables finales: {df.shape[1]}")

missing = (
    pd.DataFrame({"nMissing": df.isna().sum(), "percentage": (df.isna().sum()/len(df))*100})
    .sort_values("percentage", ascending=False)
)
print("Cantidad de varoles nulos por variable y porcentaje:")
display(missing)

print("\nRango de PublishDate:")
valid_publish_dates = df["PublishDate"].dropna()
if not valid_publish_dates.empty:
    print(valid_publish_dates.min()," - " ,valid_publish_dates.max())
else:
    print("No valid PublishDate entries found.")

Casas finales después de limpieza: 42505
Variables finales: 25
Cantidad de varoles nulos por variable y porcentaje:


,nMissing,percentage
Floor,41509,97.656746
ConservationStatus,34121,80.275262
PublishDate,24725,58.169627
NumberOfWC,22668,53.330196
LotSize,21845,51.393954
BuiltArea,20807,48.951888
NumberOfBedrooms,15383,36.191036
EnergyEfficiencyLevel,15380,36.183978
ElectricCarsCharging,15380,36.183978
Garage,15380,36.183978



Rango de PublishDate:
2018-05-02 00:00:00  -  2026-09-02 20:30:15.870000


In [71]:
# Save clean dataset

#MODIFICATION: new name to distinguish it from the original project's CSV.
OUTPUT_PATH = "portugal_listings_limpio.csv"

df.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)

print(
    f"Archivo guardado como: {OUTPUT_PATH}"
)


Archivo guardado como: portugal_listings_limpio.csv
